In [106]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
sns.set_theme(style="whitegrid", context="talk")


## Clean GDP per capita data


In [107]:
# gdp_raw = pd.read_csv("../data/gdp_per_capita/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv", skiprows=4, encoding="utf-8-sig")
gdp_raw = pd.read_csv("/Users/faresfawzi/Documents/PhD/DataViz/GSP/data/gdp_absolute/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_94769.csv", skiprows=4, encoding="utf-8-sig")
gdp_raw.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,3.092428e+09,3.276188e+09,3.346623e+09,2.471419e+09,2.880903e+09,3.324034e+09,3.834730e+09,4.265651e+09,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP (current US$),NY.GDP.MKTP.CD,2.420569e+10,2.495889e+10,2.707323e+10,3.176914e+10,3.027955e+10,3.380618e+10,...,9.780765e+11,1.020956e+12,1.018715e+12,9.386076e+11,1.114145e+12,1.228968e+12,1.179359e+12,1.242694e+12,NaN,NaN
2,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,1.875346e+10,1.805322e+10,1.879944e+10,1.995593e+10,1.426000e+10,1.449724e+10,1.715223e+10,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP (current US$),NY.GDP.MKTP.CD,1.190481e+10,1.270772e+10,1.363059e+10,1.446891e+10,1.580356e+10,1.692088e+10,...,6.940513e+11,7.778403e+11,1.026996e+12,9.637847e+11,1.026651e+12,1.063649e+12,9.382384e+11,7.363850e+11,NaN,NaN
4,Angola,AGO,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,8.437694e+10,8.951279e+10,8.073443e+10,5.885246e+10,7.955954e+10,1.312122e+11,1.071677e+11,1.009989e+11,NaN,NaN


In [108]:
year_cols = [c for c in gdp_raw.columns if c.isdigit()]

gdp_long = gdp_raw[["Country Name", "Country Code", *year_cols]].melt(
    id_vars=["Country Name", "Country Code"],
    var_name="year",
    value_name="gdp_usd"
)

gdp_long["Country Code"] = gdp_long["Country Code"].str.upper().str.strip()
gdp_long["year"] = pd.to_numeric(gdp_long["year"], errors="coerce").astype("Int64")
gdp_long["gdp_usd"] = pd.to_numeric(gdp_long["gdp_usd"], errors="coerce")

gdp_long = gdp_long.sort_values(["Country Code", "year"]).reset_index(drop=True)

# gdp_long["gdp_usd_filled"] = (
#     gdp_long.groupby("Country Code")["gdp_usd"]
#     .transform(lambda s: s.interpolate(limit_direction="both"))
# )

gdp_long.head()


,Country Name,Country Code,year,gdp_usd
0,Aruba,ABW,1960,NaN
1,Aruba,ABW,1961,NaN
2,Aruba,ABW,1962,NaN
3,Aruba,ABW,1963,NaN
4,Aruba,ABW,1964,NaN


In [109]:
gdp_long['Country Code'].nunique()

266

In [110]:
# year summary
print(f"Minimum year: {gdp_long['year'].min()}")
print(f"Maximum year: {gdp_long['year'].max()}")
print(f"Unique years: {gdp_long['year'].unique()}")
print(f"Number of unique years: {gdp_long['year'].nunique()}")

# number of unique countries
print(f"Unique countries: {gdp_long['Country Code'].nunique()}")

# number of missing values in gdp_usd before and after interpolation
gdp_missing_strict = gdp_long["gdp_usd"].isna().sum()
print(f"Missing gdp_usd before interpolation: {gdp_missing_strict:,} ({gdp_missing_strict / len(gdp_long) * 100:.2f}%)")


Minimum year: 1960
Maximum year: 2025
Unique years: <IntegerArray>
[1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972,
 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985,
 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998,
 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011,
 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024,
 2025]
Length: 66, dtype: Int64
Number of unique years: 66
Unique countries: 266
Missing gdp_usd before interpolation: 2,995 (17.06%)


In [155]:
tmp = gdp_long.sort_values(["Country Code", "year"]).copy()
tmp = tmp[(tmp["year"] >= 1960) & (tmp["year"] <= 2024)].copy()

tmp["missing"] = tmp["gdp_usd"].isna()

tmp["has_before"] = tmp.groupby("Country Code")["gdp_usd"].transform(lambda s: s.notna().cummax())
tmp["has_after"] = (
    tmp.iloc[::-1]
    .groupby("Country Code")["gdp_usd"]
    .transform(lambda s: s.notna().cummax())
    .iloc[::-1]
)

tmp["missing_start"] = tmp["missing"] & ~tmp["has_before"]
tmp["missing_middle"] = tmp["missing"] & tmp["has_before"] & tmp["has_after"]
tmp["missing_end"] = tmp["missing"] & ~tmp["has_after"]

country_missing = tmp.groupby("Country Code").agg(
    all_missing=("missing", "all"),
    start_gap=("missing_start", "any"),
    middle_gap=("missing_middle", "any"),
    end_gap=("missing_end", "any"),
).reset_index()

print("No GDP at all:", country_missing["all_missing"].sum())
print(
    "Missing only at the start:",
    ((country_missing["start_gap"]) & ~country_missing["middle_gap"] & ~country_missing["end_gap"] & ~country_missing["all_missing"]).sum()
)
print(
    "Missing in the middle:",
    ((country_missing["middle_gap"]) & ~country_missing["all_missing"]).sum()
)
print(
    "Missing only at the end:",
    ((country_missing["end_gap"]) & ~country_missing["middle_gap"] & ~country_missing["start_gap"] & ~country_missing["all_missing"]).sum()
)
print("Missing at the start and middle but not end:", ((country_missing["start_gap"]) & (country_missing["middle_gap"]) & ~country_missing["end_gap"] & ~country_missing["all_missing"]).sum())
print("Missing in the middle and end but not start:", ((country_missing["middle_gap"]) & (country_missing["end_gap"]) & ~country_missing["start_gap"] & ~country_missing["all_missing"]).sum())
print("Missing at the start and end:", ((country_missing["start_gap"]) & (country_missing["end_gap"]) & ~country_missing["all_missing"]).sum())
country_missing.head()


No GDP at all: 4
Missing only at the start: 87
Missing in the middle: 5
Missing only at the end: 1
Missing at the start and middle but not end: 3
Missing in the middle and end but not start: 0
Missing at the start and end: 21


,Country Code,all_missing,start_gap,middle_gap,end_gap
0,ABW,False,True,False,False
1,AFE,False,False,False,False
2,AFG,False,True,False,True
3,AFW,False,False,False,False
4,AGO,False,True,False,False


In [113]:
tmp = gdp_long.sort_values(["Country Code", "year"]).copy()
tmp["missing"] = tmp["gdp_usd"].isna()

tmp["has_before"] = tmp.groupby("Country Code")["gdp_usd"].transform(lambda s: s.notna().cummax())
tmp["has_after"] = (
    tmp.iloc[::-1]
    .groupby("Country Code")["gdp_usd"]
    .transform(lambda s: s.notna().cummax())
    .iloc[::-1]
)

tmp["missing_middle"] = tmp["missing"] & tmp["has_before"] & tmp["has_after"]
tmp[tmp['missing_middle']==True]

,Country Name,Country Code,year,gdp_usd,missing,has_before,has_after,missing_middle
2556,Channel Islands,CHI,2008,NaN,True,True,True,True
3722,Djibouti,DJI,1986,NaN,True,True,True,True
5826,Equatorial Guinea,GNQ,1978,NaN,True,True,True,True
5827,Equatorial Guinea,GNQ,1979,NaN,True,True,True,True
9754,St. Martin (French part),MAF,2012,NaN,True,True,True,True
9755,St. Martin (French part),MAF,2013,NaN,True,True,True,True
9757,St. Martin (French part),MAF,2015,NaN,True,True,True,True
9758,St. Martin (French part),MAF,2016,NaN,True,True,True,True
9759,St. Martin (French part),MAF,2017,NaN,True,True,True,True
9760,St. Martin (French part),MAF,2018,NaN,True,True,True,True


In [114]:
country_col = "Country Code" if "Country Code" in gdp_long.columns else "country_code"


missing_cells = int(gdp_long.isna().sum().sum())
rows_with_missing = int(gdp_long.isna().any(axis=1).sum())
missing_pct = 100 * missing_cells / (gdp_long.shape[0] * gdp_long.shape[1])




rows = [
    ("Number of entries", f"{len(gdp_long):,}"),
    ("Number of countries", f"{gdp_long['Country Code'].nunique():,}"),
    ("Number of years included", f"{gdp_long['year'].nunique():,}"),
    ("Oldest year", f"{gdp_long['year'].min()}"),
    ("Most recent year", f"{gdp_long['year'].max()}"),
]

non_null_gdp = int(gdp_long['gdp_usd'].notna().sum())
rows.append(("GDP values available", f"{non_null_gdp:,}"))

rows.append((
    "Missing data",
    f"{missing_cells:,} cells ({missing_pct:.2f}%), {rows_with_missing:,} rows with >=1 missing value"
))

md_table = ["| Metric | Value |", "|---|---|"] + [f"| {k} | {v} |" for k, v in rows]
print("\n".join(md_table))


| Metric | Value |
|---|---|
| Number of entries | 17,556 |
| Number of countries | 266 |
| Number of years included | 66 |
| Oldest year | 1960 |
| Most recent year | 2025 |
| GDP values available | 14,561 |
| Missing data | 2,995 cells (4.26%), 2,995 rows with >=1 missing value |


In [115]:
missing_by_col = gdp_long.isna().sum()
missing_by_col = missing_by_col[missing_by_col > 0].sort_values(ascending=False)

col_table = ["| Column | Missing |", "|---|---|"] + [
    f"| {col} | {int(n):,} |" for col, n in missing_by_col.items()
]
print("\n".join(col_table))


| Column | Missing |
|---|---|
| gdp_usd | 2,995 |


In [116]:
# Countries (codes) with zero GDP values in the whole series
no_gdp_mask = (
    gdp_long.groupby("Country Code")["gdp_usd"]
    .apply(lambda s: s.notna().sum() == 0)
)

no_gdp_codes = no_gdp_mask[no_gdp_mask].index.tolist()

print("How many country codes have NO GDP entries at all:", len(no_gdp_codes))
print(no_gdp_codes)


How many country codes have NO GDP entries at all: 4
['GIB', 'INX', 'PRK', 'VGB']


In [117]:
olympic_df = pd.read_csv("../data/olympic_medals.csv")

print(olympic_df.shape)
olympic_df.head()

(21261, 9)


,season,year,medal,country_code,country,games,sport,event_gender,event_name
0,Summer,1896,Gold,USA,United States,1896 Athens,Athletics,Men's,100m
1,Summer,1896,Silver,GER,Germany,1896 Athens,Athletics,Men's,100m
2,Summer,1896,Bronze,HUN,Hungary,1896 Athens,Athletics,Men's,100m
3,Summer,1896,Bronze,USA,United States,1896 Athens,Athletics,Men's,100m
4,Summer,1896,Gold,USA,United States,1896 Athens,Athletics,Men's,400m


In [118]:
missing_cells = olympic_df.isna().sum()
rows_with_missing = int(olympic_df.isna().any(axis=1).sum())
missing_pct = 100 * missing_cells / (olympic_df.shape[0] * olympic_df.shape[1])

print("Number of entries/medals", f"{len(olympic_df):,}"),
print("Number of sports", f"{olympic_df['sport'].nunique():,}"),
print("Number of countries", f"{olympic_df['country'].nunique():,}"),
print("Missing_data", rows_with_missing)

Number of entries/medals 21,261
Number of sports 92
Number of countries 174
Missing_data 0


In [119]:
olympic_df["country_code"] = olympic_df["country_code"].str.upper().str.strip()
olympic_df["year"] = pd.to_numeric(olympic_df["year"], errors="coerce").astype("Int64")

In [120]:
olympic_modern_df = olympic_df[olympic_df["year"] >= 1960].copy()

merged = olympic_modern_df.merge(
    gdp_long[["Country Code", "year", "gdp_usd"]],
    left_on=["country_code", "year"],
    right_on=["Country Code", "year"],
    how="left"
).drop(columns=["Country Code"])

print("GDP filled rate (before):", merged["gdp_usd"].notna().mean())


missing = merged.loc[merged["gdp_usd"].isna(), ["country_code", "year"]].drop_duplicates()
gdp_codes = set(gdp_long["Country Code"])

missing["reason"] = np.where(
    missing["country_code"].isin(gdp_codes),
    "code_exists_but_year_missing",
    "code_not_in_gdp"
)

print(missing["reason"].value_counts())
print("\nTop codes not in GDP:")
print(
    missing.loc[missing["reason"] == "code_not_in_gdp", "country_code"]
    .value_counts()
)


GDP filled rate (before): 0.6870752464336007
code_not_in_gdp                 385
code_exists_but_year_missing     38
Name: reason, dtype: int64

Top codes not in GDP:
NED    25
SUI    25
BUL    19
DEN    18
GER    17
       ..
PAR     1
EUN     1
IOP     1
SRI     1
BUR     1
Name: country_code, Length: 62, dtype: int64


In [121]:
import sys
sys.path.append("../src")
from utils import code_fix

In [122]:
olympic_modern_df["country_code_fix"] = olympic_modern_df["country_code"].replace(code_fix)

merged_fixed = olympic_modern_df.merge(
    gdp_long[["Country Code", "year", "gdp_usd"]],
    left_on=["country_code_fix", "year"],
    right_on=["Country Code", "year"],
    how="left"
).drop(columns=["Country Code"])

# strict value from the exact year; fallback value from interpolated country series
# merged_fixed["gdp_usd_final"] = merged_fixed["gdp_usd"].fillna(merged_fixed["gdp_usd_filled"])

print("GDP filled rate (strict):",sum(merged_fixed["gdp_usd"].notna()), "out of", merged_fixed.shape[0], ", which is", merged_fixed["gdp_usd"].notna().mean())
# print("GDP filled rate (with interpolation):", sum(merged_fixed["gdp_usd_final"].notna()), "out of", merged_fixed.shape[0], ", which is", merged_fixed["gdp_usd_final"].notna().mean())

missing_final = merged_fixed.loc[
    merged_fixed["gdp_usd"].isna(),
    ["country_code", "country_code_fix", "year"],
].drop_duplicates()

gdp_codes = set(gdp_long["Country Code"])

missing_final["reason"] = np.where(
    missing_final["country_code_fix"].isin(gdp_codes),
    "code_exists_but_year_missing",
    "code_not_in_gdp"
)

print()
print("Missing breakdown (after fix + interpolation):")
print(missing_final["reason"].value_counts())

print()
print("Top original IOC codes still missing:")
print(missing_final["country_code"].value_counts().head(20))

print()
print("Top mapped codes still missing:")
print(missing_final["country_code_fix"].value_counts().head(20))

print()
print("Sample missing rows:")
print(missing_final.head(20))


GDP filled rate (strict): 14731 out of 16333 , which is 0.9019163656401151

Missing breakdown (after fix + interpolation):
code_exists_but_year_missing    69
code_not_in_gdp                 17
Name: reason, dtype: int64

Top original IOC codes still missing:
PRK    12
TPE    12
TCH     8
YUG     8
POL     7
URS     7
ROU     7
BUL     5
CUB     3
MGL     3
HUN     2
LBN     2
EST     1
EOR     1
IOA     1
IOP     1
BWI     1
LAT     1
LTU     1
ISV     1
Name: country_code, dtype: int64

Top mapped codes still missing:
PRK    12
TPE    12
CZE     8
SRB     8
POL     7
RUS     7
ROU     7
BGR     5
CUB     3
MNG     3
HUN     2
LBN     2
EST     1
EOR     1
IOA     1
IOP     1
BWI     1
LVA     1
LTU     1
VIR     1
Name: country_code_fix, dtype: int64

Sample missing rows:
     country_code country_code_fix  year                        reason
11            BWI              BWI  1960               code_not_in_gdp
14            HUN              HUN  1960  code_exists_but_year_missing
17 

In [123]:
print("rows:", len(merged_fixed))
merged_fixed.isna().sum()

rows: 16333


season                 0
year                   0
medal                  0
country_code           0
country                0
games                  0
sport                  0
event_gender           0
event_name             0
country_code_fix       0
gdp_usd             1602
dtype: int64

In [124]:
# 1) Country-code coverage between datasets (after your mapping)
olymp_codes = set(olympic_modern_df["country_code_fix"].dropna().unique())
gdp_codes = set(gdp_long["Country Code"].dropna().unique())

olymp_not_in_gdp = sorted(olymp_codes - gdp_codes)
gdp_not_in_olymp = sorted(gdp_codes - olymp_codes)

print("Unique mapped Olympic codes:", len(olymp_codes))
print("Unique GDP codes:", len(gdp_codes))
print("Olympics codes NOT in GDP:", len(olymp_not_in_gdp))
print("GDP codes NOT in Olympics:", len(gdp_not_in_olymp))

# Which Olympic countries are not in GDP (with original IOC code + name)
olymp_missing_countries = (
    olympic_modern_df[olympic_modern_df["country_code_fix"].isin(olymp_not_in_gdp)]
    [["country", "country_code", "country_code_fix"]]
    .drop_duplicates()
    .sort_values(["country_code_fix", "country"])
)
print("\nOlympic countries not in GDP:")
print(olymp_missing_countries)

# Which GDP countries are not in Olympics (code + name)
gdp_only_countries = (
    gdp_long[gdp_long["Country Code"].isin(gdp_not_in_olymp)]
    [["Country Code", "Country Name"]]
    .drop_duplicates()
    .sort_values(["Country Code", "Country Name"])
)
print("\nGDP countries not in Olympics:")
print(gdp_only_countries)


Unique mapped Olympic codes: 147
Unique GDP codes: 266
Olympics codes NOT in GDP: 6
GDP codes NOT in Olympics: 125

Olympic countries not in GDP:
                                        country country_code country_code_fix
16297                                       AIN          AIN              AIN
4549                        British West Indies          BWI              BWI
15951                      Refugee Olympic Team          EOR              EOR
15379              Independent Olympic Athletes          IOA              IOA
9865   Independent Participants (ex Yugoslavia)          IOP              IOP
8480                             Chinese Taipei          TPE              TPE
4608                 Republic of China (Taiwan)          TPE              TPE

GDP countries not in Olympics:
      Country Code                    Country Name
0              ABW                           Aruba
66             AFE     Africa Eastern and Southern
198            AFW      Africa Western and Ce

In [125]:
print(gdp_only_countries[:50])

     Country Code                                   Country Name
0             ABW                                          Aruba
66            AFE                    Africa Eastern and Southern
198           AFW                     Africa Western and Central
264           AGO                                         Angola
396           AND                                        Andorra
462           ARB                                     Arab World
726           ASM                                 American Samoa
792           ATG                            Antigua and Barbuda
1188          BEN                                          Benin
1320          BGD                                     Bangladesh
1452          BHR                                        Bahrain
1584          BIH                         Bosnia and Herzegovina
1716          BLZ                                         Belize
1848          BOL                                        Bolivia
2112          BTN        

In [126]:
print(gdp_only_countries[50:100])

      Country Code                                       Country Name
7260           INX                                     Not classified
8118           KHM                                           Cambodia
8184           KIR                                           Kiribati
8250           KNA                                St. Kitts and Nevis
8448           LAC  Latin America & Caribbean (excluding high income)
8514           LAO                                            Lao PDR
8646           LBR                                            Liberia
8712           LBY                                              Libya
8844           LCN                          Latin America & Caribbean
8910           LDC       Least developed countries: UN classification
8976           LIC                                         Low income
9174           LMC                                Lower middle income
9240           LMY                                Low & middle income
9306           LSO  

In [127]:
print(gdp_only_countries[100:])

      Country Code                                       Country Name
14058          SOM                                 Somalia, Fed. Rep.
14190          SSA         Sub-Saharan Africa (excluding high income)
14256          SSD                                        South Sudan
14322          SSF                                 Sub-Saharan Africa
14388          SST                                       Small states
14454          STP                              Sao Tome and Principe
14784          SWZ                                           Eswatini
14850          SXM                          Sint Maarten (Dutch part)
14916          SYC                                         Seychelles
15048          TCA                           Turks and Caicos Islands
15114          TCD                                               Chad
15180          TEA         East Asia & Pacific (IDA & IBRD countries)
15246          TEC       Europe & Central Asia (IDA & IBRD countries)
15576          TLA  

In [128]:
merged_fixed["year"].min()

1960

In [129]:
merged_fixed["year"].max()

2024

In [130]:
merged_fixed["year"].unique()

<IntegerArray>
[1960, 1964, 1968, 1972, 1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008,
 2012, 2016, 2024, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022, 2020]
Length: 25, dtype: Int64

In [131]:
total_entries = len(merged_fixed)
num_sports = merged_fixed["sport"].nunique()
num_countries = merged_fixed["country"].nunique()

num_summer_games = merged_fixed.loc[merged_fixed["season"] == "Summer", "games"].nunique()
num_winter_games = merged_fixed.loc[merged_fixed["season"] == "Winter", "games"].nunique()
num_total_games = merged_fixed["games"].nunique()

# years present in medals table
medal_years = merged_fixed["year"].nunique()

# years where GDP is available (strict and final)
gdp_years_strict = merged_fixed.loc[merged_fixed["gdp_usd"].notna(), "year"].nunique()
gdp_years_final = merged_fixed.loc[merged_fixed["gdp_usd"].notna(), "year"].nunique()

# GDP missing stats
gdp_missing_strict = merged_fixed["gdp_usd"].isna().sum()
# gdp_missing_final = merged_fixed["gdp_usd_final"].isna().sum()
gdp_missing_strict_pct = 100 * gdp_missing_strict / total_entries
# gdp_missing_final_pct = 100 * gdp_missing_final / total_entries

summary_rows = [
    ("Number of entries/medals", f"{total_entries:,}"),
    ("Number of sports", f"{num_sports:,}"),
    ("Number of countries", f"{num_countries:,}"),
    ("Number of Summer Olympics Events", f"{num_summer_games:,}"),
    ("Number of Winter Olympics Events", f"{num_winter_games:,}"),
    ("Missing data (GDP)", f"({gdp_missing_strict_pct:.2f}%)"),
]




md_table = ["| Variable | Value |", "|---|---|"] + [f"| {k} | {v} |" for k, v in summary_rows]
print("\n".join(md_table))


| Variable | Value |
|---|---|
| Number of entries/medals | 16,333 |
| Number of sports | 80 |
| Number of countries | 166 |
| Number of Summer Olympics Events | 17 |
| Number of Winter Olympics Events | 17 |
| Missing data (GDP) | (9.81%) |
